In [1]:
import os
import numpy as np
import pandas as pd
import scanpy as sc
from scipy import sparse

In [2]:
# ============================================================
# SETTINGS
# ============================================================

H5AD_PATH = "/scratch/ts4594/RDS_files/Final_Final/ATAC/tala_atac_peak_matrix_annotated.h5ad"
OUTDIR = "LPSvsUnstim_edgeR_pseudobulk_ATAC_Cytotoxic_T-NKcells"
os.makedirs(OUTDIR, exist_ok=True)

DONOR_COL = "donor_id"
CELLTYPE_COL = "morocco_Cluster_celltypes_consensus"
CELLTYPE_LABEL = "Cytotoxic T/NK cells"

STIM_COL = "Stimulation"
SEX_COL = "Sex"
LIFE_COL = "Lifestyle"
AGE_COL = "Age"

COUNTS_LAYER = None
MIN_CELLS_PER_DONOR = 10

In [3]:
# ============================================================
# LOAD
# ============================================================

adata = sc.read_h5ad(H5AD_PATH)
print(adata)

required = [
    DONOR_COL, CELLTYPE_COL, STIM_COL,
    SEX_COL, LIFE_COL, AGE_COL
]

missing = [x for x in required if x not in adata.obs.columns]
if missing:
    raise ValueError(f"Missing metadata columns: {missing}")

AnnData object with n_obs × n_vars = 167801 × 214587
    obs: 'batch_id', 'donor_id', 'library_batch', 'predicted.celltype.l2', 'predicted.celltype.l1', 'Cluster_celltypes', 'Sample.ID', 'Pool', 'Batch', 'Lifestyle', 'Sex', 'Age', 'Ethnicity', 'Stimulation', 'modality', 'gsi', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', '_indices', '_scvi_batch', '_scvi_labels', 'leiden_multivi', 'age', 'morocco_Cluster_celltypes', 'morocco_Sub_Cluster_celltypes', 'morocco_Cluster_celltypes_knn_transfer', 'morocco_Cluster_celltypes_knn_confidence', 'morocco_Cluster_celltypes_knn_transfer_thresholded', 'morocco_Cluster_celltypes_knn_source', 'morocco_Cluster_celltypes_graph_transfer', 'morocco_Cluster_celltypes_graph_purity', 'morocco_Cluster_celltypes_graph_transfer_thresholded', 'morocco_Cluster_celltypes_graph_iter', 'morocco_Cluster_cell

In [4]:
for x in adata.obs["morocco_Cluster_celltypes_consensus"].unique():
    print(x)

Memory T cells
Cytotoxic T/NK cells
B cells
Naive CD8 T cells
Naive CD4 T cells
Monocytes
Stressed
nan
Dendritic cells


In [5]:
# ============================================================
# SUBSET Cytotoxic_T-NKcells cells
# ============================================================

adata = adata[
    adata.obs[CELLTYPE_COL].astype(str) == CELLTYPE_LABEL
].copy()

print("Cytotoxic_T-NKcellsobject:")
print(adata)

Cytotoxic_T-NKcellsobject:
AnnData object with n_obs × n_vars = 49022 × 214587
    obs: 'batch_id', 'donor_id', 'library_batch', 'predicted.celltype.l2', 'predicted.celltype.l1', 'Cluster_celltypes', 'Sample.ID', 'Pool', 'Batch', 'Lifestyle', 'Sex', 'Age', 'Ethnicity', 'Stimulation', 'modality', 'gsi', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', '_indices', '_scvi_batch', '_scvi_labels', 'leiden_multivi', 'age', 'morocco_Cluster_celltypes', 'morocco_Sub_Cluster_celltypes', 'morocco_Cluster_celltypes_knn_transfer', 'morocco_Cluster_celltypes_knn_confidence', 'morocco_Cluster_celltypes_knn_transfer_thresholded', 'morocco_Cluster_celltypes_knn_source', 'morocco_Cluster_celltypes_graph_transfer', 'morocco_Cluster_celltypes_graph_purity', 'morocco_Cluster_celltypes_graph_transfer_thresholded', 'morocco_Cluster_celltypes_graph_it

In [6]:

# ============================================================
# GET COUNTS
# ============================================================

if COUNTS_LAYER is None:
    X = adata.X
else:
    X = adata.layers[COUNTS_LAYER]

if sparse.issparse(X):
    X = X.tocsr()
else:
    X = sparse.csr_matrix(X)

In [7]:
# ============================================================
# SAVE PEAK ANNOTATIONS
# ============================================================

peak_annot = adata.var.copy()
peak_annot = peak_annot.reset_index().rename(columns={"index": "peak_id"})

# Add peak_id explicitly from var_names
peak_annot["peak_id"] = adata.var_names.astype(str)

# Try to standardize best-gene column if present
possible_best_gene_cols = [
    "best_gene",
    "BestGene",
    "bestGene",
    "gene",
    "Gene",
    "nearest_gene",
    "NearestGene",
    "peak_gene",
    "PeakGene"
]

found_gene_cols = [c for c in possible_best_gene_cols if c in peak_annot.columns]

if len(found_gene_cols) > 0:
    peak_annot["best_gene"] = peak_annot[found_gene_cols[0]].astype(str)
else:
    peak_annot["best_gene"] = NA = pd.NA
    print("WARNING: No best-gene column found in adata.var.")
    print("Available adata.var columns:")
    print(list(peak_annot.columns))

peak_annot_file = os.path.join(
    OUTDIR,
    "Cytotoxic_T-NKcells_ATAC_peak_annotations.tsv"
)

peak_annot.to_csv(
    peak_annot_file,
    sep="\t",
    index=False
)

print("Saved peak annotations:")
print(peak_annot_file)

Saved peak annotations:
LPSvsUnstim_edgeR_pseudobulk_ATAC_Cytotoxic_T-NKcells/Cytotoxic_T-NKcells_ATAC_peak_annotations.tsv


In [8]:
# ============================================================
# PREPARE METADATA
# ============================================================

obs = adata.obs.copy()

obs = obs.dropna(
    subset=[
        DONOR_COL, STIM_COL,
        SEX_COL, LIFE_COL, AGE_COL
    ]
)

adata = adata[obs.index].copy()
X = X[adata.obs_names.isin(obs.index), :]

obs = adata.obs.copy()

for col in [DONOR_COL, STIM_COL, SEX_COL, LIFE_COL]:
    obs[col] = obs[col].astype(str)

obs[STIM_COL] = obs[STIM_COL].replace({
    "unstim": "Unstim",
    "UNSTIM": "Unstim",
    "lps": "LPS",
    "LPS": "LPS"
})

# IMPORTANT: include Stimulation so LPS and Unstim stay separate
obs["pb_sample_id"] = (
    obs[DONOR_COL].astype(str) + "__" +
    obs[LIFE_COL].astype(str) + "__" +
    obs[SEX_COL].astype(str) + "__" +
    obs[STIM_COL].astype(str)
)

In [9]:
# ============================================================
# PSEUDOBULK BY DONOR × LIFE × SEX × STIMULATION
# ============================================================

pb_ids = obs["pb_sample_id"].values
unique_pb_ids = pd.Index(pd.unique(pb_ids))

pb_counts = []
pb_meta_rows = []

for pb in unique_pb_ids:
    idx = np.where(pb_ids == pb)[0]
    n_cells = len(idx)

    if n_cells < MIN_CELLS_PER_DONOR:
        continue

    summed = np.asarray(X[idx, :].sum(axis=0)).ravel()
    sub = obs.iloc[idx]

    pb_counts.append(summed)

    pb_meta_rows.append({
        "pb_sample_id": pb,
        "donor_id": sub[DONOR_COL].iloc[0],
        "Stimulation": sub[STIM_COL].iloc[0],
        "Sex": sub[SEX_COL].iloc[0],
        "Lifestyle": sub[LIFE_COL].iloc[0],
        "Age": sub[AGE_COL].iloc[0],
        "n_cells": n_cells,
        "library_size": summed.sum()
    })

pb_counts = np.vstack(pb_counts)
pb_meta = pd.DataFrame(pb_meta_rows)

counts_df = pd.DataFrame(
    pb_counts.T,
    index=adata.var_names.astype(str),
    columns=pb_meta["pb_sample_id"].astype(str)
)

In [10]:
# ============================================================
# SAVE FULL PSEUDOBULK
# ============================================================

counts_df.to_csv(
    os.path.join(OUTDIR, "Cytotoxic_T-NKcells_ATAC_pseudobulk_counts_all.tsv"),
    sep="\t"
)

pb_meta.to_csv(
    os.path.join(OUTDIR, "Cytotoxic_T-NKcells_ATAC_pseudobulk_metadata_all.tsv"),
    sep="\t",
    index=False
)

In [11]:
# ============================================================
# SAVE FOUR LPS vs UNSTIM COMPARISON MATRICES
# ============================================================

comparisons = [
    ("URBAN", "Male"),
    ("URBAN", "Female"),
    ("RURAL", "Male"),
    ("RURAL", "Female")
]

for lifestyle, sex in comparisons:

    tag = f"{lifestyle}_{sex}"

    meta_sub = pb_meta[
        (pb_meta["Lifestyle"].astype(str) == lifestyle) &
        (pb_meta["Sex"].astype(str) == sex)
    ].copy()

    meta_sub = meta_sub[
        meta_sub["Stimulation"].isin(["Unstim", "LPS"])
    ].copy()

    sample_ids = meta_sub["pb_sample_id"].astype(str).tolist()
    counts_sub = counts_df[sample_ids].copy()

    counts_sub.to_csv(
        os.path.join(OUTDIR, f"Cytotoxic_T-NKcells_ATAC_counts_LPSvsUnstim_{tag}.tsv"),
        sep="\t"
    )

    meta_sub.to_csv(
        os.path.join(OUTDIR, f"Cytotoxic_T-NKcells_ATAC_metadata_LPSvsUnstim_{tag}.tsv"),
        sep="\t",
        index=False
    )

    print("\n", tag)
    print("counts:", counts_sub.shape)
    print("metadata:", meta_sub.shape)
    print(meta_sub["Stimulation"].value_counts())


 URBAN_Male
counts: (214587, 56)
metadata: (56, 8)
Stimulation
Unstim    28
LPS       28
Name: count, dtype: int64

 URBAN_Female
counts: (214587, 40)
metadata: (40, 8)
Stimulation
Unstim    20
LPS       20
Name: count, dtype: int64

 RURAL_Male
counts: (214587, 50)
metadata: (50, 8)
Stimulation
Unstim    25
LPS       25
Name: count, dtype: int64

 RURAL_Female
counts: (214587, 42)
metadata: (42, 8)
Stimulation
Unstim    21
LPS       21
Name: count, dtype: int64


In [12]:
# ============================================================
# SAVE SUMMARY
# ============================================================

summary = (
    pb_meta
    .groupby(["Lifestyle", "Sex", "Stimulation"])
    .agg(
        n_donors=("donor_id", "nunique"),
        median_cells=("n_cells", "median"),
        median_library_size=("library_size", "median")
    )
    .reset_index()
)

summary.to_csv(
    os.path.join(OUTDIR, "Cytotoxic_T-NKcells_ATAC_pseudobulk_summary.tsv"),
    sep="\t",
    index=False
)

print("\nDone.")


Done.
